# Compiled Lagrangian operators

`model.lagrangian()` returns a `CompiledLagrangian`. Operators act on that
object: field replacements, spacetime derivatives, integration by parts, and
Symbolica export. Gauge variation and BRST are in `gauge_and_brst.ipynb`.


## Setup


In [1]:
import re
import sys
from fractions import Fraction
from pathlib import Path

from symbolica import Expression, S

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ANSI_ESCAPE_RE = re.compile(r"\x1B\[[0-?]*[ -/]*[@-~]")


def clean(text):
    return ANSI_ESCAPE_RE.sub("", str(text))


def show(title, result):
    print("==========")
    print(title)
    if isinstance(result, dict):
        print(f"{len(result)} vertex signature(s)")
        print()
        for signature, expression in result.items():
            print("Vertex:", signature)
            print("Rule:", clean(expression))
            print()
    else:
        print(clean(result))
        print()


def show_model(model, *fields, compact_form=None, sum_notation=None, simplify=True):
    source_terms = model.lagrangian_decl.source_terms
    if source_terms:
        lagrangian_source = (
            sum(source_terms[1:], source_terms[0])
            if len(source_terms) > 1
            else source_terms[0]
        )
        show("Lagrangian", lagrangian_source)
    lagrangian = model.lagrangian()
    if fields:
        show("Feynman Rule", lagrangian.feynman_rule(*fields, include_delta=False, simplify=simplify))
    else:
        show("Feynman Rules", lagrangian.feynman_rule(include_delta=False, simplify=simplify))
    if compact_form is not None:
        show("Compact Form", compact_form)
    if sum_notation is not None:
        show("Sum Notation", sum_notation)

from dataclasses import replace

from feynpy import Model, PartialD, dirac_field, flavor_index, scalar_field
from lagrangian.operator_action import (
    FieldOperator,
    OperatorExpansionError,
    TermOperator,
    partial,
    replacement_operator,
    single_field_result,
)


In [2]:
mu, f = S("mu"), S("f")
Generation = flavor_index("Generation", 3, prefix="f")
Phi = scalar_field("Phi", self_conjugate=True)
Chi = scalar_field("Chi", self_conjugate=True)
Omega = scalar_field("Omega", self_conjugate=True)
H = scalar_field("H", self_conjugate=True)
lepton = dirac_field(
    "l",
    class_members=("e", "mu", "ta"),
    indices=(Generation,),
    flavor_index=Generation,
)

model_phi3 = Model(Phi * Phi * Phi + Phi * PartialD(Phi, mu))
L_phi3 = model_phi3.lagrangian()
show_model(model_phi3)


Lagrangian
Phi * Phi * Phi + Phi * PartialD(Phi, mu)

Feynman Rules
2 vertex signature(s)

Vertex: ('Phi', 'Phi', 'Phi')
Rule: 6𝑖

Vertex: ('Phi', 'Phi')
Rule: pcomp(q1,mu1_int)+pcomp(q2,mu1_int)



## Symbolica export

`to_symbolica()` turns the compiled terms into an ordinary Symbolica
expression. Flavor classes can be expanded at export time.


In [3]:
show("coordinate-free export", L_phi3.to_symbolica())
show("derivative with respect to Phi", L_phi3.to_symbolica().derivative(S("Phi")))

flavor_model = Model(S("g") * lepton.bar(f) * lepton(f) * H)
show("compact export", flavor_model.lagrangian().to_symbolica())
show("flavor-expanded export", flavor_model.lagrangian().to_symbolica(flavor_expand=True))


coordinate-free export
Phi*PartialD(Phi,mu)+Phi^3

derivative with respect to Phi
Phi*der(1,0,PartialD(Phi,mu))+PartialD(Phi,mu)+3*Phi^2

compact export
-H*g*l(f,i_decl_1)*lbar(f,i_decl_1)

flavor-expanded export
-H*g*mu(i_decl_1)*mubar(i_decl_1)-H*g*ebar(i_decl_1)*e(i_decl_1)-H*g*tabar(i_decl_1)*ta(i_decl_1)



## Field and term operators

`FieldOperator` acts slot by slot. `TermOperator` acts once per compiled
term. `replacement_operator` is the common special case `Phi → Chi`.


In [4]:
replace_phi = replacement_operator("Phi_to_Chi", {Phi: Chi()})
show("Phi → Chi", L_phi3.apply_operator(replace_phi).to_symbolica())

prepend_each = FieldOperator(
    "prepend_each",
    on_field=lambda occ: None if occ.field is not Phi else single_field_result((Omega(), occ)),
)
prepend_once = TermOperator(
    "prepend_once",
    apply_to_term=lambda term: (replace(term, fields=(Omega(),) + term.fields),),
)
show("FieldOperator: Omega once per Phi slot", L_phi3.apply_operator(prepend_each).to_symbolica())
show("TermOperator: Omega once per term", L_phi3.apply_operator(prepend_once).to_symbolica())

scale = TermOperator("scale", apply_to_term=lambda term: (replace(term, coupling=2 * term.coupling),))
show("replace then scale", L_phi3.apply_operators(replace_phi, scale).to_symbolica())


Phi → Chi
Phi*PartialD(Chi,mu)+Chi*PartialD(Phi,mu)+3*Phi^2*Chi

FieldOperator: Omega once per Phi slot
2*Phi*Omega*PartialD(Phi,mu)+Phi^2*PartialD(Omega,mu)+3*Phi^3*Omega

TermOperator: Omega once per term
Phi*Omega*PartialD(Phi,mu)+Phi^3*Omega

replace then scale
2*Phi*PartialD(Chi,mu)+2*Chi*PartialD(Phi,mu)+6*Phi^2*Chi



\
## Integration by parts

Modulo a total derivative,
$$
\phi^2\,\Box\phi \simeq -2\,\phi\,(\partial_\mu\phi)(\partial_\mu\phi).
$$
The scalar IBP normal form makes that identity executable.


In [5]:
c = S("c")
L1 = Model(c * Phi * Phi * PartialD(PartialD(Phi, mu), mu)).lagrangian()
L2 = Model(-2 * c * Phi * PartialD(Phi, mu) * PartialD(Phi, mu)).lagrangian()
Delta = Model(
    c * Phi * Phi * PartialD(PartialD(Phi, mu), mu)
    + 2 * c * Phi * PartialD(Phi, mu) * PartialD(Phi, mu)
).lagrangian()

show("L1", L1.to_symbolica())
show("L2", L2.to_symbolica())
show("IBP(L1)", L1.ibp_normal_form().to_symbolica())
show("IBP(L2)", L2.ibp_normal_form().to_symbolica())
show("IBP(L1 - L2)", Delta.ibp_normal_form().to_symbolica())


L1
Phi^2*c*PartialD(PartialD(Phi,mu),mu)

L2
-2*Phi*c*PartialD(Phi,mu)^2

IBP(L1)
2/3*Phi*c*PartialD(Phi,mu)^2+4/3*Phi^2*c*PartialD(PartialD(Phi,mu),mu)

IBP(L2)
2/3*Phi*c*PartialD(Phi,mu)^2+4/3*Phi^2*c*PartialD(PartialD(Phi,mu),mu)

IBP(L1 - L2)
0



## Spacetime derivative `partial(mu)`

`partial(mu)` creates a fresh derivative action on every acted slot. Use
`on=Phi` to restrict the Leibniz expansion.


In [6]:
show("partial(mu)[L]", L_phi3.apply_operator(partial(mu)).to_symbolica())

product = Model(Phi * Chi).lagrangian()
show("partial(mu)[Phi Chi]", product.apply_operator(partial(mu)).to_symbolica())
show("partial(mu, on=Phi)[Phi Chi]", product.apply_operator(partial(mu, on=Phi)).to_symbolica())

try:
    product.apply_operator(partial(mu), max_generated_terms=1)
except OperatorExpansionError as exc:
    show("expansion cap", exc)


partial(mu)[L]
Phi*PartialD(PartialD(Phi,mu),mu)+3*Phi^2*PartialD(Phi,mu)+PartialD(Phi,mu)^2

partial(mu)[Phi Chi]
Phi*PartialD(Chi,mu)+Chi*PartialD(Phi,mu)

partial(mu, on=Phi)[Phi Chi]
Chi*PartialD(Phi,mu)

expansion cap
Operator expansion exceeds configured limit (operator='d_mu', slot=1, replacement_len=1, derivative_count_on_slot=0, projected_terms=1, max_generated_terms=1).

